In [1]:
%load_ext autoreload
%autoreload 2

In [5]:
import sys
sys.path.insert(0, "..")


from app_config import AppConfig

from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel


/home/dinhln1/Desktop/hcmut_master/is_assignment/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load CLIP model

In [6]:

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 49645.59it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Image inputs and compute similarity
texts = ["a photo of a dog", "a photo of a cat"]

text_inputs = processor(text=texts, return_tensors="pt", padding=True)

In [ ]:
text_encoded_1 = [1,2,3]
text_encodede_2 = [1,3,4,5,6]

#Batching
0: padding
[1,2,3,0,0]
[1,3,4,5,6]

(5,2)

In [20]:
text_inputs

{'input_ids': tensor([[49406,   320,  1125,   539,   320,  1929, 49407],
        [49406,   320,  1125,   539,   320,  2368, 49407]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1]])}

In [22]:
text_features = model.get_text_features(**text_inputs)

In [25]:
text_features.pooler_output.shape

torch.Size([2, 512])

In [27]:
def compute_text_similarity(texts: list[str]):
    text_inputs = processor(text=texts, return_tensors="pt", padding=True)
    text_features = model.get_text_features(**text_inputs).pooler_output
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
    similarity = text_features @ text_features.T
    return similarity

In [33]:
texts = ["a dog", "a woman", "golden", "husky"]
similarity = compute_text_similarity(texts)
similarity

tensor([[1.0000, 0.8956, 0.8427, 0.8246],
        [0.8956, 1.0000, 0.8548, 0.7554],
        [0.8427, 0.8548, 1.0000, 0.7335],
        [0.8246, 0.7554, 0.7335, 1.0000]], grad_fn=<MmBackward0>)

In [34]:
# Not good at text similarity, let's try text-image similarity

In [40]:
from PIL import Image

label = {
    "dog": "images/dog.webp",
    "cat": "images/cat.webp",
    "robot": "images/robot.webp",
}

inverted_label = {
    "0": "dog",
    "1": "cat",
    "2": "robot",
}

images = [Image.open(path) for path in label.values()]

def compute_text_image_similarity(texts: list[str], images: list[Image.Image]):
    text_inputs = processor(text=texts, return_tensors="pt", padding=True)
    image_inputs = processor(images=images, return_tensors="pt")
    text_features = model.get_text_features(**text_inputs).pooler_output
    image_features = model.get_image_features(**image_inputs).pooler_output
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    similarity = text_features @ image_features.T
    return similarity


In [46]:
q1 = compute_text_image_similarity(["a dog"], images)

q1


tensor([[0.2601, 0.2301, 0.2016]], grad_fn=<MmBackward0>)

In [47]:
q2 = compute_text_image_similarity(["a cat"], images)

q2

tensor([[0.2047, 0.2832, 0.1978]], grad_fn=<MmBackward0>)

In [48]:
q3 = compute_text_image_similarity(["Robotics"], images)

q3

tensor([[0.1602, 0.1822, 0.2810]], grad_fn=<MmBackward0>)